In [0]:
# CELL 1: Install & Verify Libraries
# -------------------------------------------------------------
# # 1. Install the library
%pip install beautifulsoup4

# 2. Verify installation immediately
try:
    from bs4 import BeautifulSoup
    print("✅ BeautifulSoup4 is installed and working correctly.")
except ImportError:
    print("❌ Error: BeautifulSoup4 failed to install. Please re-run this cell.")


In [0]:
# CELL 2: Setup & Dynamic Link Finder
# -------------------------------------------------------------------------
import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime, timedelta

# 1. Define URL
base_page_url = "https://dubaipulse.gov.ae"

# -------------------------------------------------------------------------
# 2. Azure Storage Paths (PASTE YOUR INFO BELOW)
# -------------------------------------------------------------------------
storage_account_name = "REMOVED_FOR_SECURITY" 
storage_account_key = "REMOVED_FOR_SECURITY"  
# <-- Paste your long access key string here

# Define Folders
landing_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/landing/"
archive_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/archive/"

# 3. Dynamic Link Finder
def get_dynamic_download_url():
    print(f"🔍 Scraping {base_page_url}...")
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    response = requests.get(base_page_url, headers=headers)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Logic: Find link containing 'download' and 'transactions.csv'
    for link in soup.find_all('a', href=True):
        href = link['href']
        if "download" in href and "transactions.csv" in href:
            final_url = f"https://dubaipulse.gov.ae{href}" if href.startswith("/") else href
            print(f"✅ Found dynamic link: {final_url}")
            return final_url
    
    raise Exception("❌ Could not find the download link on the page.")

print("Configuration Loaded & Native Token Engine Initialized.")


In [0]:
import os
import time
import requests

def download_snapshot_with_retry(max_retries=3):
    local_path = "/tmp/current.csv"
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    # Check network reachability first
    try:
        fresh_url = get_dynamic_download_url()
    except Exception as e:
        print("❌ Cannot reach the website from this cluster due to Databricks Serverless network locks.")
        print("💡 Follow the MANUAL FIX below to skip this block and keep moving forward!")
        return

    for attempt in range(1, max_retries + 1):
        try:
            print(f"⬇️ Attempt {attempt}/{max_retries}: Downloading to driver node...")
            
            with requests.get(fresh_url, headers=headers, stream=True, timeout=30) as r:
                r.raise_for_status()
                with open(local_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=10*1024*1024):
                        if chunk:
                            f.write(chunk)
            
            print(" ✅ Download complete.")
            break 
            
        except Exception as e:
            print(f" ⚠️ Attempt {attempt} failed: {str(e)}")
            if attempt < max_retries:
                print(" ⏳ Waiting 5 seconds...")
                time.sleep(5)
            else:
                print("❌ All retries failed.")
                raise e
    
    # Corrected Serverless Cloud Storage copy approach
    if os.path.exists(local_path):
        print(f"⬆️ Uploading to Bronze Landing Zone...")
        try:
            import fsspec
            storage_options = {"account_name": storage_account_name, "account_key": storage_account_key}
            with fsspec.open(f"{landing_zone}current.csv", "wb", **storage_options) as target_file:
                with open(local_path, "rb") as source_file:
                    target_file.write(source_file.read())
            os.remove(local_path)
            print("✅ Success! 'current.csv' is updated in the Data Lake.")
        except Exception as upload_err:
            print(f"❌ Storage upload failed: {upload_err}")

# Run it
download_snapshot_with_retry()

In [0]:
# CELL 3: Retention Policy (Serverless Adapted)
# -------------------------------------------------------------------------
import fsspec

def cleanup_old_files(days_to_keep=14):
    print("🧹 Checking archive for old files...")
    cutoff_date = datetime.now() - timedelta(days=days_to_keep)
    
    storage_options = {
        "account_name": storage_account_name,
        "account_key": storage_account_key
    }
    
    try:
        # Open connection to the archive folder path natively
        fs = fsspec.filesystem("abfs", **storage_options)
        archive_path_stripped = archive_zone.replace("abfss://", "").replace("dfs.core.windows.net", "")
        
        if fs.exists(archive_path_stripped):
            files = fs.ls(archive_path_stripped)
            for file_path in files:
                filename = file_path.split("/")[-1].replace(".csv", "")
                try:
                    file_date = datetime.strptime(filename, "%Y-%m-%d")
                    if file_date < cutoff_date:
                        print(f" 🗑️ Deleting old archive file: {file_path}")
                        fs.rm(file_path)
                except ValueError:
                    continue
        else:
            print(" ℹ️ Archive directory does not exist yet.")
    except Exception as e:
        print(f" ℹ️ Archive is empty or cannot be accessed yet: {e}")
 
# Run it
cleanup_old_files()


In [0]:
# CELL 4: Archival Logic (Serverless Adapted)
# -------------------------------------------------------------------------
import fsspec

def archive_current_file():
    storage_options = {
        "account_name": storage_account_name,
        "account_key": storage_account_key
    }
    
    # Strip paths to format expected by native blob driver
    landing_path_raw = landing_zone.replace("abfss://", "").replace("dfs.core.windows.net", "") + "current.csv"
    today_str = datetime.now().strftime("%Y-%m-%d")
    archive_path_raw = archive_zone.replace("abfss://", "").replace("dfs.core.windows.net", "") + f"{today_str}.csv"
    
    try:
        fs = fsspec.filesystem("abfs", **storage_options)
        
        # Check if current.csv exists in landing zone
        if fs.exists(landing_path_raw):
            print(f"📦 Archiving previous 'current.csv' to: {archive_zone}{today_str}.csv")
            fs.mv(landing_path_raw, archive_path_raw)
        else:
            print("ℹ️ No 'current.csv' found in landing folder. First manual run?")
            
    except Exception as e:
        print(f"ℹ️ Archival step skipped or handled manually: {e}")
 
# Run it
archive_current_file()


In [0]:
# CELL 1: Install & Verify Libraries
# -------------------------------------------------------------
%pip install beautifulsoup4 adlfs

try:
    from bs4 import BeautifulSoup
    import adlfs
    print("✅ All required local and cloud storage libraries are installed correctly.")
except ImportError:
    print("❌ Error: Missing library dependencies.")